## 模型
### 访问模型


In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(model="gemini-2.5-flash", model_provider="openai")

# print(model.invoke("月亮的首都是哪里？"))

stream = model.stream("月亮的首都是哪里？")

for chunk in model.stream("月亮的首都是哪里？"):
    print(chunk.content, end="", flush=True)


月亮并没有真正的首都，因为它是一个自然星球，没有政府和人类社会。

但如果从神话传说或诗意的角度来看，很多人会想到**月宫**（或广寒宫）。这是中国神话中嫦娥居住的地方，被想象成月亮上最美丽、最核心的宫殿。

所以，在想象中，月宫就是月亮的首都吧！

### Agent中使用模型

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
model = init_chat_model(model="gemini-2.5-flash", model_provider="openai")

agent = create_agent(model=model)

# response = agent.invoke({
#     "messages": [{"role": "user", "content": "月亮的首都是哪里？"}]
# })
# print(response)

for token, metadata in agent.stream(
        {"messages": [{"role": "user", "content": "月亮的首都是哪里？"}]},
        stream_mode="messages"
):
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token


要注意，Agent的stream模式同样返回一个generator，但是其结构由stream_mode参数决定：
- messages: 返回LLM生成的每一个片段，是一个包含token和metadata的元组（Tuple）
- updates: 返回Agent运行过程中的每一次事件，例如与LLM的对话、工具的调用等
- custom: 返回通过stream writer记录的每一次自定义的输出

如果是为了流式输出AI返回的结果，使用messages模式即可。

## 消息（Messages）

我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

load_dotenv()
model = init_chat_model(model="gemini-2.5-flash", model_provider="openai")
agent = create_agent(model=model)

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好，我是虎哥"),
        AIMessage(content="你好，虎哥，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？")
    ]
})

for message in response['messages']:
    message.pretty_print()

## 多模态消息
### 在线图片

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

load_dotenv()

model = init_chat_model(model="gemini-2.5-flash", model_provider="openai")
agent = create_agent(model=model)

multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])

for token, metadata in agent.stream({
    "messages": [multimodal_message]
}, stream_mode="messages"):
    if token.content:
        print(token.content, end="", flush=True)


### 本地图片
我们需要将图片数据转换成base64字符串，然后发送给模型。

In [ ]:
import base64

# 例如，有一个用户上传的文件，是字节格式img_bytes，我们先将其进行base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

# 调用Agent，发送消息
response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## 提示词（Prompts）

发送给大模型的所有消息都可以称为提示词（Prompt），它直接影响模型的输出结果。

其中，SystemMessage尤为重要，我们把SystemMessage称为系统提示词（System Prompt），它可以给模型设定角色和本次聊天的背景，对模型生成的内容有很大的影响。

通过优化Prompt从而让模型输出更理想的结果的这一过程，我们称为提示词工程（Prompt Engineering）。
### 系统提示词

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="You are a science fiction writer, create a capital city at the users request."
)

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="月球的首都在哪里?")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)

### 结构化输出

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from pydantic import BaseModel

load_dotenv()

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

# 然后，我们就可以创建智能体并设置结构化输出的格式了。
model = init_chat_model(model="gemini-2.5-flash", model_provider="openai")

agent = create_agent(model=model,
                     system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
                     response_format=CapitalInfo)  # 设置结构化输出的格式
response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)

city = response['structured_response']

print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")


## 工具（Tools）

智能体在工作时，需要将函数的名称、输入、作用传递给大模型，默认情况下这些信息的来源是：
- 工具名称：函数名
- 工具输入：函数入参
- 工具作用：函数的注释

如果要覆盖工具的入参信息则会复杂很多，我们要借助于Pydantic或JSON约束。

In [ ]:
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field

load_dotenv()

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

model = init_chat_model(model="gemini-2.5-flash", model_provider="openai")

agent = create_agent(model=model,
                     system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
                     tools=[tool1, get_weather])  # 设置结构化输出的格式

# 如果采用stream模式的updates模式，可以看到工具调用的具体步骤
for chunk in agent.stream(
        {"messages": [HumanMessage(content="467、529的平方根是多少?")]},
        stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")
        print()

for token, metadata in agent.stream(
        {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
        stream_mode="messages"
):
    print(token.content, end="", flush=True)